In [ ]:
# Cài pyspark
!pip install -q transformers sentence-transformers faiss-cpu pyarrow peft

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import ndcg_score
import faiss

In [ ]:
# ==========================================
# SEED & DEVICE SETUP
# ==========================================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import zipfile
import os
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/ReSys_Train/Data"
EXTRACT_BASE = "/content/extracted_data"

files_config = {
    "train": "Software_train.parquet-20260517T163339Z-3-001.zip",
    "val": "Software_val.parquet-20260517T163340Z-3-001.zip",
    "test": "Software_test.parquet-20260517T163337Z-3-001.zip",
    "item": "Software_item.parquet-20260517T163250Z-3-001.zip"
}

dfs = {}

for name, filename in files_config.items():
    zip_path = os.path.join(DATA_PATH, filename)
    extract_folder = os.path.join(EXTRACT_BASE, name)

    if not os.path.exists(zip_path):
        print(f"❌ Không tìm thấy file zip của {name} tại: {zip_path}")
        continue

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

    extracted_files = os.listdir(extract_folder)

    parquet_targets = [f for f in extracted_files if "parquet" in f.lower() or not f.startswith(".")]

    if not parquet_targets:
        print(f"⚠️ Giải nén thành công nhưng không tìm thấy file dữ liệu bên trong thư mục {extract_folder}")
        continue

    final_parquet_path = os.path.join(extract_folder, parquet_targets[0])

    dfs[name] = pd.read_parquet(final_parquet_path)

train_df = dfs.get("train")
val_df   = dfs.get("val")
test_df  = dfs.get("test")
item_df  = dfs.get("item")

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# =====================================================================
# Ô CHẠY TỔNG HỢP: KHỞI TẠO BIẾN NỀN + ĐỊNH NGHĨA HÀM SIÊU TỐC
# =====================================================================

# 1. Đảm bảo safe() và item_text_map đã sẵn sàng
def safe(x):
    if pd.isna(x): return "None"
    return str(x)

item_text_map = {}
for r in item_df.itertuples():
    item_text_map[r.parent_asin] = (
        f"{safe(r.semantic_text)}\n"
        f"Category: {safe(r.main_category)}\n"
        f"Store: {safe(r.store)}\n"
        f"Price: {safe(r.price)}\n"
        f"Average Rating: {safe(r.average_rating)}\n"
        f"Rating Count: {safe(r.rating_number)}"
    )

# 2. Khởi tạo chính xác base_model và tokenizer (Nơi gây ra lỗi của ông)
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import torch

# Đảm bảo các biến MODEL_NAME và device của ông đã chạy ở các ô trước đó nhé
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

# 3. Hàm Mean Pooling nền
def mean_pooling(out, mask):
    token = out.last_hidden_state
    mask = mask.unsqueeze(-1).expand(token.size()).float()
    return (token * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)


# 4. Hàm Bước 1: Mã hóa trải nghiệm hàng loạt bằng GPU theo Batch
def precompute_user_experience_vectors(df, desc="Mã hóa trải nghiệm"):
    titles = df['title'].fillna('None').astype(str)
    texts = df['text'].fillna('None').astype(str)
    base_texts = df['parent_asin'].map(item_text_map).fillna('None')

    combined_strings = (
        base_texts + "\n" +
        "User Review Title: " + titles + "\n" +
        "User Review Content: " + texts
    ).tolist()

    all_vectors = []
    batch_size = 256

    with torch.no_grad():
        for i in tqdm(range(0, len(combined_strings), batch_size), desc=desc):
            batch_strs = combined_strings[i:i+batch_size]

            enc = tokenizer(batch_strs, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            out = base_model(**enc) # <--- Chỗ này giờ đã có base_model ở trên định nghĩa sẵn!
            embs = F.normalize(mean_pooling(out, enc["attention_mask"]), dim=1).cpu().numpy()

            all_vectors.append(embs)

    return np.vstack(all_vectors)

# 5. Hàm Bước 2: Gom nhóm và tích hợp trọng số thời gian trên CPU
def aggregate_user_history(df, experience_vectors):
    user_vector_hist = {}
    df['_temp_vector'] = list(experience_vectors)

    for uid, g in df.groupby("user_id"):
        g = g.reset_index(drop=True)
        n = len(g)

        vectors = np.array(g['_temp_vector'].tolist())
        idx_array = np.arange(n)
        recency = np.exp(-(n - idx_array) / max(n, 1))
        ratings = g['rating'].fillna(0).to_numpy()
        weights = (recency * (1 + (ratings / 5.0))).reshape(-1, 1)

        user_combined_vector = np.sum(vectors * weights, axis=0) / np.sum(weights)
        user_combined_vector = user_combined_vector / (np.linalg.norm(user_combined_vector) + 1e-9)

        user_vector_hist[uid] = user_combined_vector

    df.drop(columns=['_temp_vector'], inplace=True, errors='ignore')
    return user_vector_hist

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# =====================================================================
# THỰC THI CHẠY SIÊU TỐC (LEAVE-LAST-OUT BATCH VERSION)
# =====================================================================
# 1. Tính toán trước cho tập Train
train_exp_vectors = precompute_user_experience_vectors(train_df, desc="Mã hóa tập Train")
train_histories = aggregate_user_history(train_df, train_exp_vectors)

# Tập Validation kế thừa trực tiếp từ lịch sử tập Train (Theo đúng logic)
val_histories = train_histories

# 2. Tính toán cho tập Test (Gộp Train + Val)
combined_train_val_df = pd.concat([train_df, val_df], ignore_index=True)

# Để tiết kiệm thời gian, ta tận dụng lại vector của tập Train đã tính, chỉ tính thêm phần của tập Val thôi!
val_exp_vectors = precompute_user_experience_vectors(val_df, desc="Mã hóa bổ sung tập Val")
combined_exp_vectors = np.vstack([train_exp_vectors, val_exp_vectors])

# Gom nhóm lại để làm history cho tập Test
test_histories = aggregate_user_history(combined_train_val_df, combined_exp_vectors)

print("--- ✔ ĐÃ HOÀN THÀNH TỔNG HỢP LỊCH SỬ ---")

In [ ]:
len(train_histories), len(val_histories), len(test_histories)

In [ ]:
import joblib

# Đường dẫn đến file đã lưu
save_history_path_train = "/content/drive/MyDrive/Colab Notebooks/ReSys_Train/Data/train_histories.joblib"
save_history_path_val   = "/content/drive/MyDrive/Colab Notebooks/ReSys_Train/Data/val_histories.joblib"
save_history_path_test  = "/content/drive/MyDrive/Colab Notebooks/ReSys_Train/Data/test_histories.joblib"

# Load lại dictionary
train_histories = joblib.load(save_history_path_train)
val_histories = joblib.load(save_history_path_val)
test_histories = joblib.load(save_history_path_test)

In [ ]:
len(train_histories), len(val_histories), len(test_histories)

(146396, 146396, 146396)

In [ ]:
class RecVectorDataset(Dataset):
    def __init__(self, df, user_vector_histories):
        self.df = df
        self.user_vector_histories = user_vector_histories

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        # Lấy ra vector 384 chiều của user (mặc định vector 0 nếu không tìm thấy)
        user_vector = self.user_vector_histories.get(r.user_id, np.zeros(384))
        item_text = item_text_map.get(r.parent_asin, "None")

        return {
            "user_vector": torch.tensor(user_vector, dtype=torch.float32),
            "item_text": item_text,
            "target": r.parent_asin
        }

train_dataset = RecVectorDataset(train_df, train_histories)
val_dataset   = RecVectorDataset(val_df, val_histories)
test_dataset  = RecVectorDataset(test_df, test_histories)

def collate_vector_fn(batch):
    # Gom các vector user lại thành tensor 2D: (Batch_size, 384)
    user_vectors = torch.stack([b["user_vector"] for b in batch])
    item_texts = [b["item_text"] for b in batch]
    targets = [b["target"] for b in batch]

    # Chỉ tokenize cho nhánh Item mục tiêu thôi
    item_tokens = tokenizer(
        item_texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    return {
        "user_vectors": user_vectors,
        "item_tokens": item_tokens,
        "target": targets
    }

In [ ]:
# =====================================================================
# DATALOADER SỬA LỖI CÚ PHÁP
# =====================================================================
batch_s = 256  # Thử nghiệm với batch size to hơn để tăng tốc độ train

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_s,
    shuffle=True,
    collate_fn=collate_vector_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_s,
    shuffle=False,
    collate_fn=collate_vector_fn,  # <--- Đã sửa sạch lỗi dính chữ ở đây
    num_workers=2,
    pin_memory=True
)

# Thêm luôn test_loader để sau này đánh giá cuối cùng cho tiện ông nhé
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_s,
    shuffle=False,
    collate_fn=collate_vector_fn,
    num_workers=2,
    pin_memory=True
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class TwoTowerVectorModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()

        # =====================================================================
        # 1. Tháp User nâng cấp: Thêm BatchNorm và Dropout để tối ưu không gian Vector
        # =====================================================================
        self.user_projection = nn.Sequential(
            nn.Linear(384, 512),
            nn.BatchNorm1d(512),   # Ổn định Gradient, giúp hội tụ nhanh hơn
            nn.ReLU(),
            nn.Dropout(0.5),       # Chống hiện tượng học vẹt (Overfitting)
            nn.Linear(512, 384)    # Trả về không gian 384 chiều để khớp với Item
        )

        # 2. Tháp Item: Giữ nguyên đóng băng hoàn toàn (Đã tối ưu bộ nhớ)
        self.item_encoder = AutoModel.from_pretrained(model_name)
        for param in self.item_encoder.parameters():
            param.requires_grad = False

    def mean_pooling(self, out, mask):
        token = out.last_hidden_state
        mask = mask.unsqueeze(-1).expand(token.size()).float()
        return (token * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward_user(self, user_vectors):
        # Chạy trực tiếp qua mạng MLP cải tiến và chuẩn hóa đầu ra L2
        emb = self.user_projection(user_vectors)
        return F.normalize(emb, dim=1) # BẮT BUỘC chuẩn hóa để tính Cosine Similarity với Item

    def forward_item(self, tokens):
        with torch.no_grad():
            out = self.item_encoder(**tokens)
            emb = self.mean_pooling(out, tokens["attention_mask"])
        return F.normalize(emb, dim=1) # Chuẩn hóa đầu ra L2 cho Item

# Khởi tạo mô hình
model = TwoTowerVectorModel(MODEL_NAME).to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Duyệt danh sách ID mặt hàng cố định
item_ids = list(item_text_map.keys())


In [ ]:
class InfoNCE(nn.Module):
    def __init__(self, temperature=0.07): # 0.07 là con số vàng trong Recommender System
        super().__init__()
        self.temperature = temperature
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, user_out, item_out):
        # user_out: (Batch, 384), item_out: (Batch, 384)
        # Tính ma trận tương đồng Cosine giữa mọi User và mọi Item trong Batch
        similarity_matrix = torch.matmul(user_out, item_out.T) / self.temperature

        # Nhãn đúng (Ground Truth) là đường chéo của ma trận (User i đi với Item i)
        labels = torch.arange(user_out.size(0), device=user_out.device)

        return self.cross_entropy(similarity_matrix, labels)

# Khởi tạo lại hàm loss mới
loss_fn = InfoNCE(temperature=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=10e-5, weight_decay=5e-2)

In [ ]:
def compute_loss(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Loss"):
            # CHỈ lấy item_tokens dạng text
            item_tokens = {k: v.to(device) for k, v in batch["item_tokens"].items()}
            # Đổi sang lấy user_vectors trực tiếp từ DataLoader mới
            user_vectors = batch["user_vectors"].to(device)

            with torch.amp.autocast('cuda'):
                item_out = model.forward_item(item_tokens)
                user_out = model.forward_user(user_vectors)
                loss = loss_fn(user_out, item_out)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def compute_metrics(model, loader, item_ids, device):
    model.eval()
    all_user_embeddings = []

    item_id_to_idx = {pid: idx for idx, pid in enumerate(item_ids)}

    targets = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Users"):
            user_vectors = batch["user_vectors"].to(device)
            with torch.amp.autocast('cuda'):
                user_out = model.forward_user(user_vectors)
            all_user_embeddings.append(user_out.cpu())

            for tgt in batch["target"]:
                targets.append(item_id_to_idx.get(tgt, -1))

        all_user_embeddings = torch.cat(all_user_embeddings, dim=0)
        targets = torch.tensor(targets, dtype=torch.long)

        # =====================================================================
        # SỬA RỦI RO 1: Dùng chính tháp Item của model để sinh Vector đồng bộ
        # =====================================================================
        print("--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---")
        all_items_pool = []
        item_batch_size = 512

        for i in range(0, len(item_ids), item_batch_size):
            batch_pids = item_ids[i:i+item_batch_size]
            batch_texts = [item_text_map.get(pid, "None") for pid in batch_pids]

            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            with torch.amp.autocast('cuda'):
                # Gọi hàm forward_item của chính model thay vì base_model gốc
                embs = model.forward_item(enc)
            all_items_pool.append(embs.cpu())

        all_item_embeddings = torch.cat(all_items_pool, dim=0)

        # =====================================================================
        # ĐOẠN TÍNH TOÁN THEO BATCH USER (Logic toán giữ nguyên vì đã chuẩn)
        # =====================================================================
        print("--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---")
        r10_list, r50_list = [], []
        n10_list, n50_list = [], []

        user_batch_size = 1000
        num_users = all_user_embeddings.size(0)
        all_item_embeddings_gpu = all_item_embeddings.to(device)

        for i in range(0, num_users, user_batch_size):
            user_chunk = all_user_embeddings[i:i+user_batch_size].to(device)
            target_chunk = targets[i:i+user_batch_size].to(device)

            similarity_scores = torch.matmul(user_chunk, all_item_embeddings_gpu.T)

            max_k = 50
            _, topk_indices = torch.topk(similarity_scores, k=max_k, dim=1)

            # So sánh vị trí trúng tuyển
            hit_positions = (topk_indices == target_chunk.view(-1, 1))

            # Nếu target bị gán bằng -1 (lỗi lệch data), loại trừ không tính hit
            invalid_mask = (target_chunk == -1).view(-1, 1)
            hit_positions = hit_positions & (~invalid_mask)

            r10_list.extend(hit_positions[:, :10].any(dim=1).float().cpu().numpy())
            r50_list.extend(hit_positions.any(dim=1).float().cpu().numpy())

            hit_ranks = torch.full((topk_indices.size(0),), -1, dtype=torch.long, device=device)
            non_zero_rows, non_zero_cols = torch.where(hit_positions)
            hit_ranks[non_zero_rows] = non_zero_cols
            hit_ranks_np = hit_ranks.cpu().numpy()

            for rank in hit_ranks_np:
                if rank != -1 and rank < 10:
                    n10_list.append(1.0 / np.log2(rank + 2))
                else:
                    n10_list.append(0.0)

                if rank != -1 and rank < 50:
                    n50_list.append(1.0 / np.log2(rank + 2))
                else:
                    n50_list.append(0.0)

            del similarity_scores, topk_indices, hit_positions, hit_ranks
            torch.cuda.empty_cache()

        del all_item_embeddings_gpu
        torch.cuda.empty_cache()

    return {
        "recall@10": np.mean(r10_list),
        "recall@50": np.mean(r50_list),
        "ndcg@10": np.mean(n10_list),
        "ndcg@50": np.mean(n50_list)
    }

In [ ]:
# =====================================================================
# TRAINING LOOP WITH EARLY STOPPING (CRITERION: VAL LOSS)
# =====================================================================
PATIENCE = 1                      # Nới lỏng lên 3 để mô hình có cơ hội vượt qua các vùng bão hòa
best_val_loss = float('inf')      # Khởi tạo giá trị vô cùng lớn cho loss
patience_counter = 0
best_model_path = "/content/drive/MyDrive/Colab Notebooks/ReSys_Train/Data/best_model.pt"

ePoch = 5
log_df = pd.DataFrame()

# Khởi tạo Scaler theo cú pháp chuẩn mới nhất hiện nay
scaler = torch.amp.GradScaler('cuda')

for epoch in range(ePoch):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Train {epoch}"):
        item_tokens = {k: v.to(device) for k, v in batch["item_tokens"].items()}

        optimizer.zero_grad()

        # Bọc các bước Forward pass vào autocast để kích hoạt Mixed Precision
        with torch.amp.autocast('cuda'):
            # 1. Nhánh Item chạy trước (trả về embedding tĩnh không có gradient)
            item_out = model.forward_item(item_tokens)

            # 2. Nhánh User chạy sau: Lấy vector trực tiếp từ batch
            user_vectors = batch["user_vectors"].to(device)
            user_out = model.forward_user(user_vectors)

            # 3. Tính loss InfoNCE dựa trên User (Dynamic) và Item (Frozen)
            loss = loss_fn(user_out, item_out)

        # Backward và Step thông qua Scaler để tránh lỗi tràn số khi dùng FP16
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # Validation Loss & Metrics
    val_loss = compute_loss(model, val_loader, loss_fn, device)
    val_metrics = compute_metrics(model, val_loader, item_ids, device)

    # =====================================================================
    # ĐOẠN SỬA: ĐỔI TIÊU CHÍ STOPPING SANG CHECK VAL LOSS GIẢM
    # =====================================================================
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        improved = f"✔ saved best model (val_loss decreased to {val_loss:.4f})"
    else:
        patience_counter += 1
        improved = f"⏳ patience {patience_counter}/{PATIENCE}"

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **val_metrics
    }
    log_df = pd.concat([log_df, pd.DataFrame([row])], ignore_index=True)

    print("\n" + "=" * 40)
    print(f"Epoch {epoch+1}")
    print("=" * 40)
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    for k, v in val_metrics.items():
        print(f"{k}: {v:.4f}")
    print(improved)

    if patience_counter >= PATIENCE:
        print("\n🛑 Early stopping triggered")
        break

Extracting Users: 100%|██████████| 572/572 [05:34<00:00,  1.71it/s]


--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---
--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---

Epoch 1
Train Loss: 3.6168
Val Loss:   5.1875
recall@10: 0.0140
recall@50: 0.0431
ndcg@10: 0.0061
ndcg@50: 0.0124
✔ saved best model (val_loss decreased to 5.1875)


Extracting Users: 100%|██████████| 572/572 [05:26<00:00,  1.75it/s]


--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---
--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---

Epoch 2
Train Loss: 3.3495
Val Loss:   5.1790
recall@10: 0.0140
recall@50: 0.0435
ndcg@10: 0.0061
ndcg@50: 0.0124
✔ saved best model (val_loss decreased to 5.1790)


Extracting Users: 100%|██████████| 572/572 [05:20<00:00,  1.78it/s]


--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---
--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---

Epoch 3
Train Loss: 3.3229
Val Loss:   5.1755
recall@10: 0.0141
recall@50: 0.0429
ndcg@10: 0.0061
ndcg@50: 0.0123
✔ saved best model (val_loss decreased to 5.1755)


Extracting Users: 100%|██████████| 572/572 [05:28<00:00,  1.74it/s]


--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---
--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---

Epoch 4
Train Loss: 3.3138
Val Loss:   5.1692
recall@10: 0.0141
recall@50: 0.0434
ndcg@10: 0.0061
ndcg@50: 0.0124
✔ saved best model (val_loss decreased to 5.1692)


Extracting Users: 100%|██████████| 572/572 [05:25<00:00,  1.76it/s]


--- Đang mã hóa toàn bộ kho Item phục vụ tính toán Metric ---
--- Đang tính toán Recall & NDCG theo từng cụm User tránh tràn VRAM ---

Epoch 5
Train Loss: 3.3076
Val Loss:   5.1649
recall@10: 0.0142
recall@50: 0.0436
ndcg@10: 0.0062
ndcg@50: 0.0125
✔ saved best model (val_loss decreased to 5.1649)
